In [1]:
import json
import time
from pathlib import Path

import pandas as pd

from nba_api.live.nba.endpoints import playbyplay as live_playbyplay
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.live.nba.endpoints import boxscore as live_boxscore


PROJECT_ROOT = "/Users/manishkavuri/Desktop/nba-lineup-decision-engine"

RAW_PBP_DIR = Path(PROJECT_ROOT) / "data/raw/pbp"
RAW_PBP_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_OUTPUT_DIR = Path(PROJECT_ROOT) / "data/processed/test_5_games_lineup_stints"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEASON = "2024-25"
SEASON_TYPE = "Regular Season"
N_GAMES = 5


def parse_clock_seconds_remaining(clock):
    """
    Parses NBA ISO-like clock strings like PT11M42.00S into seconds remaining.
    """
    import re

    if pd.isna(clock):
        return None

    clock = str(clock)

    m = re.match(r"PT(?:(\d+)M)?(?:(\d+(?:\.\d+)?)S)?", clock)

    if not m:
        return None

    minutes = float(m.group(1) or 0)
    seconds = float(m.group(2) or 0)

    return minutes * 60 + seconds


def fetch_pbp_json_from_nba(game_id):
    game_id = str(game_id).zfill(10)
    pbp = live_playbyplay.PlayByPlay(game_id=game_id)
    return pbp.get_dict()


def load_pbp_actions_dataframe(game_id, force_refresh=False):
    game_id = str(game_id).zfill(10)
    local_path = RAW_PBP_DIR / f"{game_id}.json"

    if local_path.exists() and not force_refresh:
        with open(local_path, "r") as f:
            raw = json.load(f)
    else:
        print(f"Downloading play-by-play for {game_id}...")
        raw = fetch_pbp_json_from_nba(game_id)

        with open(local_path, "w") as f:
            json.dump(raw, f)

        print(f"Saved raw PBP to {local_path}")

    actions = raw.get("game", {}).get("actions", [])

    if not actions:
        raise ValueError(f"No play-by-play actions found for game {game_id}")

    df = pd.DataFrame(actions)

    expected_cols = [
        "actionNumber",
        "period",
        "clock",
        "actionType",
        "description",
        "teamId",
        "personId",
        "scoreHome",
        "scoreAway",
    ]

    for col in expected_cols:
        if col not in df.columns:
            df[col] = None

    return df

In [2]:
test_game_id = "0022400062"

pbp_test = load_pbp_actions_dataframe(test_game_id)

print(pbp_test.shape)
display(pbp_test.head())

Saved raw PBP to /Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0022400062.json
(564, 56)


,actionNumber,clock,timeActual,period,periodType,actionType,subType,qualifiers,personId,x,...,officialId,foulPersonalTotal,foulTechnicalTotal,foulDrawnPlayerName,foulDrawnPersonId,blockPlayerName,blockPersonId,turnoverTotal,stealPlayerName,stealPersonId
0,2,PT12M00.00S,2024-10-23T02:06:26.0Z,1,REGULAR,period,start,[],0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4,PT11M55.00S,2024-10-23T02:06:29.9Z,1,REGULAR,jumpball,recovered,[],201144,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7,PT11M32.00S,2024-10-23T02:06:53.1Z,1,REGULAR,3pt,Jump Shot,[],1630162,67.460578,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,8,PT11M29.00S,2024-10-23T02:06:56.0Z,1,REGULAR,rebound,defensive,[],2544,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9,PT11M18.00S,2024-10-23T02:07:06.9Z,1,REGULAR,3pt,Jump Shot,[],1629060,30.141261,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
from nba_api.stats.endpoints import leaguegamefinder

def get_season_game_ids(season="2024-25", season_type="Regular Season", n_games=5):
    finder = leaguegamefinder.LeagueGameFinder(
        season_nullable=season,
        season_type_nullable=season_type
    )

    games = finder.get_data_frames()[0].copy()

    # GAME_ID can appear twice because each game has two teams
    games["GAME_ID"] = games["GAME_ID"].astype(str).str.zfill(10)

    games = games.sort_values("GAME_DATE")

    game_ids = (
        games["GAME_ID"]
        .drop_duplicates()
        .head(n_games)
        .tolist()
    )

    return game_ids

In [5]:
game_ids = get_season_game_ids(
    season=SEASON,
    season_type=SEASON_TYPE,
    n_games=5
)

print(game_ids)

for game_id in game_ids:
    print(f"\nTesting download for {game_id}")
    pbp = load_pbp_actions_dataframe(game_id)
    print(game_id, pbp.shape)

['0022400062', '0022400061', '0022400064', '0022400069', '0022400070']

Testing download for 0022400062
0022400062 (564, 56)

Testing download for 0022400061
Saved raw PBP to /Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0022400061.json
0022400061 (453, 56)

Testing download for 0022400064
Saved raw PBP to /Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0022400064.json
0022400064 (612, 56)

Testing download for 0022400069
Saved raw PBP to /Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0022400069.json
0022400069 (586, 56)

Testing download for 0022400070
Saved raw PBP to /Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0022400070.json
0022400070 (652, 56)


In [6]:
from pathlib import Path
import pandas as pd

RAW_PBP_DIR = Path("/Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp")

downloaded_files = list(RAW_PBP_DIR.glob("*.json"))

sizes = []

for file in downloaded_files:
    sizes.append({
        "file": file.name,
        "size_mb": file.stat().st_size / (1024 ** 2)
    })

size_df = pd.DataFrame(sizes)

display(size_df)

avg_size_mb = size_df["size_mb"].mean()
total_downloaded_mb = size_df["size_mb"].sum()

estimated_regular_season_games = 1230
estimated_full_season_raw_mb = avg_size_mb * estimated_regular_season_games
estimated_full_season_raw_gb = estimated_full_season_raw_mb / 1024

print(f"Downloaded files: {len(size_df)}")
print(f"Average raw PBP file size: {avg_size_mb:.3f} MB")
print(f"Current raw PBP storage: {total_downloaded_mb:.3f} MB")
print(f"Estimated full regular season raw storage: {estimated_full_season_raw_mb:.2f} MB")
print(f"Estimated full regular season raw storage: {estimated_full_season_raw_gb:.2f} GB")

,file,size_mb
0,0042500111.json,0.318473
1,0022400062.json,0.401051
2,0022400069.json,0.415113
3,0022400064.json,0.435317
4,0022400070.json,0.459057
5,0022400061.json,0.324598


Downloaded files: 6
Average raw PBP file size: 0.392 MB
Current raw PBP storage: 2.354 MB
Estimated full regular season raw storage: 482.49 MB
Estimated full regular season raw storage: 0.47 GB
